In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# C2-T1 first real long-terminal validation

One 181-frame terminal is generated once and shared by OFF/message-0/message-1. This notebook is intentionally executable: its only runtime entry is the cancellable subprocess below. Do not run it locally; it is the authorized Colab GPU path.


In [ ]:
from pathlib import Path
import sys, subprocess, json
REPOSITORY_URL = 'https://github.com/RICHAAARC/SC-SSTW.git'
SOURCE_BRANCH = 'c2a-2a-colab-preparation'
SOURCE = Path('/content/c2t1_source')
if SOURCE.exists():
    existing_remote = subprocess.check_output(['git', '-C', str(SOURCE), 'remote', 'get-url', 'origin'], text=True).strip()
    if existing_remote != REPOSITORY_URL:
        raise RuntimeError(f'existing source has unexpected origin: {existing_remote}')
else:
    subprocess.run(['git', 'init', str(SOURCE)], check=True)
    subprocess.run(['git', '-C', str(SOURCE), 'remote', 'add', 'origin', REPOSITORY_URL], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'fetch', '--depth', '1', 'origin', SOURCE_BRANCH], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'checkout', '--detach', '--force', 'FETCH_HEAD'], check=True)
print('Source:', subprocess.check_output(['git', '-C', str(SOURCE), 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', 'diffusers', 'transformers', 'accelerate', 'ftfy', 'sentencepiece', 'safetensors', 'huggingface_hub', 'numpy', 'Pillow'], check=True)
subprocess.run(['ffmpeg', '-version'], check=True)
# Keep Colab CUDA PyTorch; runtime versions are persisted in the run record.


## Fixed scope

The package writes all 45 ordinary groups after the special group, 3 groups per 15 logical states. It persists one shared terminal, three pre-codec RGB tensors, seven MP4 inputs, actual FP32 writing evidence, and exactly 28 independently cache-reset VAE re-encodes. Normal/cut/speed lengths are 181/178/145. The two edit branches derive by decoding each corresponding saved normal MP4 and applying one further fixed lossy encode; conclusions therefore concern joint edit-plus-reencoding conditions.


In [ ]:
from datetime import datetime, timezone
CONFIG = SOURCE / 'runtime/c2t1/c2t1_first_run.json'
RUN_ID = 'c2t1_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUTPUT = Path('/content/drive/MyDrive/Video-WM/C2T1') / RUN_ID
print(CONFIG.read_text())
print('Output:', OUTPUT)
if OUTPUT.exists():
    raise FileExistsError(str(OUTPUT))


In [ ]:
import os, signal
command = [sys.executable, '-u', '-m', 'runtime.c2t1.run', '--config', str(CONFIG), '--output', str(OUTPUT)]
# Keep runner ownership of OUTPUT; its parent is a stable Drive location for an interruption-safe launcher log.
LOG = OUTPUT.parent / f'{RUN_ID}.launcher.log'
LOG.parent.mkdir(parents=True, exist_ok=True)
with LOG.open('w', encoding='utf-8') as log:
    process = subprocess.Popen(command, cwd=SOURCE, start_new_session=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
        returncode = process.wait()
    except BaseException:
        try: process.send_signal(signal.SIGTERM)
        except ProcessLookupError: pass
        try: process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            try: os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError: pass
            process.wait()
        raise
print('launcher exit', returncode)
print((OUTPUT / 'result.json').read_text() if (OUTPUT / 'result.json').exists() else 'No result file')
print('Drive log:', LOG)
if returncode: raise subprocess.CalledProcessError(returncode, command)


## Reading boundary

The receiver re-encodes only `1 + 4*n` frames after each cache-reset start. Incomplete tails are retained as missing rather than completed observations. Search uses only the 28 saved VAE observations: `a={0.8,1,1.25}`, predeclared integer `b`, and `g=0..3`; it neither performs local edit paths nor uses message/attack truth to filter candidates. Results retain every candidate and failure, report alignment-equivalence classes, score gaps, coverage and ambiguity, and have no FPR or PASS claim.
